# CIFAR-10 Image Classification Project
## Computer Vision mit Transfer Learning

**Ziel**: Klassifikation von 10 verschiedenen Objektkategorien mit CNNs

## 1. Imports und Setup

In [ ]:
# Grundlegende Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TensorFlow und Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.utils import to_categorical

# ============================================
# M1 MAX GPU OPTIMIERUNGEN (METAL)
# ============================================

print('='*60)
print('M1 MAX METAL GPU SETUP')
print('='*60)

# 1. GPU-Geräte erkennen und konfigurieren
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Memory Growth aktivieren (verhindert dass GPU den ganzen VRAM reserviert)
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'✅ Metal GPU gefunden: {len(gpus)} GPU(s)')
        print(f'   {gpus[0]}')
    except RuntimeError as e:
        print(f'⚠️  Fehler bei GPU-Konfiguration: {e}')
else:
    print('⚠️  Keine GPU gefunden - läuft auf CPU')

# 2. Mixed Precision für bis zu 2x schnelleres Training
# (Nutzt float16 statt float32 wo möglich)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print('✅ Mixed Precision aktiviert (float16)')

# 3. XLA JIT Compiler - DEAKTIVIERT für TensorFlow Metal
# XLA ist nicht kompatibel mit Metal GPU in TensorFlow 2.16
# tf.config.optimizer.set_jit(True)  # Deaktiviert
print('⚠️  XLA JIT Compiler deaktiviert (nicht kompatibel mit Metal)')

# 4. Threading optimieren (für Data Loading Pipeline)
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

# Einstellungen
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('='*60)
print('SYSTEM INFORMATION')
print('='*60)
print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU verfügbar: {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Alle Geräte: {tf.config.list_physical_devices()}')
print(f'Mixed Precision Policy: {mixed_precision.global_policy().name}')
print('='*60)
print('🚀 M1 Max Metal GPU Optimierungen aktiviert!')
print('   Erwarteter Speedup: 2-4x schneller als CPU (ohne XLA)')
print('='*60)

## 2. Daten laden und vorbereiten

In [ ]:
# CIFAR-10 Dataset laden
print("Lade CIFAR-10 Dataset...")
(train_images, train_labels), (test_images, test_labels) = cifar10.load_data()

print(f"\nOriginal Trainingsbilder: {train_images.shape}")
print(f"Original Testbilder: {test_images.shape}")
print(f"Label Shape: {train_labels.shape}")

In [ ]:
# 🚀 M1 MAX EMPFEHLUNG: Mit Metal GPU und 32GB RAM kannst du
# problemlos alle 50.000 Bilder nutzen für beste Accuracy!

# Optionen:
# n = 10000   # Schneller Test (~2-3 Minuten Training)
# n = 25000   # Mittel (~5-7 Minuten Training)
n = 50000     # ⭐ EMPFOHLEN für M1 Max (~10-15 Minuten, beste Accuracy!)

train_images = train_images[:n]
train_labels = train_labels[:n]

print(f"Trainingsbilder: {train_images.shape}")
print(f"Labels: {train_labels.shape}")
print(f"\n✅ Nutze {n:,} Bilder - optimiert für M1 Max!")
if n == 50000:
    print("   🎯 Alle Trainingsbilder werden genutzt!")
    print("   📈 Erwarte höhere Accuracy (~75-80%)")

In [ ]:
# Klassennamen definieren
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Klassen im Dataset:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

## 3. Explorative Datenanalyse (EDA)

In [ ]:
# Label-Verteilung analysieren
train_labels_flat = train_labels.flatten()

# Zählen
label_counts = pd.Series(train_labels_flat).value_counts().sort_index()
print("\nLabel-Verteilung:")
for idx, count in label_counts.items():
    print(f"  {class_names[idx]:12s} (Klasse {idx}): {count:4d} Bilder")

# Visualisierung
plt.figure(figsize=(12, 5))
bars = plt.bar(range(10), [label_counts[i] for i in range(10)])
plt.xticks(range(10), class_names, rotation=45, ha='right')
plt.xlabel('Klasse')
plt.ylabel('Anzahl')
plt.title('Verteilung der Klassen im Trainingsdatensatz')
plt.grid(axis='y', alpha=0.3)

# Farben für Balken
for bar in bars:
    bar.set_alpha(0.8)

plt.tight_layout()
plt.show()

print(f"\nDer Datensatz ist {'fast ausgewogen' if label_counts.std() < 100 else 'unausgewogen'}")

In [ ]:
# Beispielbilder visualisieren
def plot_sample_images(images, labels, class_names, n_samples=25):
    """
    Zeigt ein Grid von Beispielbildern mit ihren Labels
    """
    plt.figure(figsize=(15, 15))
    for i in range(n_samples):
        plt.subplot(5, 5, i + 1)
        plt.imshow(images[i])
        plt.title(class_names[labels[i][0]], fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Zufällige Beispiele zeigen
print("Beispielbilder aus dem Trainingsdatensatz:")
random_indices = np.random.choice(len(train_images), 25, replace=False)
plot_sample_images(train_images[random_indices], 
                   train_labels[random_indices], 
                   class_names)

In [ ]:
# Datenstatistiken
print("\n=== Datenstatistiken ===")
print(f"Bildgröße: {train_images.shape[1]}x{train_images.shape[2]} Pixel")
print(f"Farbkanäle: {train_images.shape[3]} (RGB)")
print(f"Pixelwert-Bereich: {train_images.min()} - {train_images.max()}")
print(f"Datentyp: {train_images.dtype}")
print(f"Durchschnittliche Pixelintensität: {train_images.mean():.2f}")

## 4. Daten normalisieren

Neuronale Netze arbeiten besser mit normalisierten Daten (0-1 Bereich)

In [ ]:
# Normalisierung: Pixelwerte von 0-255 auf 0-1 skalieren
train_images_normalized = train_images.astype('float32') / 255.0
test_images_normalized = test_images.astype('float32') / 255.0

print("Nach Normalisierung:")
print(f"Neuer Wertebereich Training: {train_images_normalized.min():.3f} - {train_images_normalized.max():.3f}")
print(f"Neuer Wertebereich Test: {test_images_normalized.min():.3f} - {test_images_normalized.max():.3f}")

In [ ]:
# Labels von 2D auf 1D reduzieren
train_labels_flat = train_labels.flatten()
test_labels_flat = test_labels.flatten()

print(f"Label Shape vorher: {train_labels.shape}")
print(f"Label Shape nachher: {train_labels_flat.shape}")

## 4.5 Data Augmentation & GPU-optimierte Pipeline

**Data Augmentation** erweitert den Datensatz künstlich durch Transformationen:
- Verhindert Overfitting
- Verbessert Generalisierung
- Simuliert verschiedene Aufnahmewinkel und -bedingungen

**tf.data Pipeline** optimiert die Datenverarbeitung für GPU:
- Parallel Prefetching (lädt Daten während GPU rechnet)
- Caching für schnellere Epochen
- Optimale Batch-Verarbeitung

In [ ]:
# 🚀 GPU-OPTIMIERTE DATA PIPELINE mit Data Augmentation

# Data Augmentation Layer (on-GPU für beste Performance!)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),  # ±10% Rotation
    layers.RandomZoom(0.1),      # ±10% Zoom
    layers.RandomTranslation(0.1, 0.1),  # ±10% Translation
], name="data_augmentation")

print("✅ Data Augmentation konfiguriert:")
print("   - Random Horizontal Flip")
print("   - Random Rotation (±10%)")
print("   - Random Zoom (±10%)")
print("   - Random Translation (±10%)")

# Visualisiere Data Augmentation
print("\n📸 Beispiel: Augmentierte Bilder")
plt.figure(figsize=(15, 3))
sample_image = train_images_normalized[0:1]  # Erstes Bild
for i in range(8):
    augmented = data_augmentation(sample_image, training=True)
    plt.subplot(1, 8, i + 1)
    plt.imshow(augmented[0])
    plt.axis('off')
plt.suptitle('Gleiche Bild mit verschiedenen Augmentationen', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("💡 Diese Transformationen passieren während des Trainings")
print("   automatisch auf der GPU - kein Overhead!")
print("="*60)

## 5. Modell aufbauen mit Transfer Learning

Wir nutzen **ResNet50**, ein vortrainiertes Modell auf ImageNet

In [ ]:
# Basis-Modell laden (ResNet50)
print("Lade vortrainiertes ResNet50-Modell...")

base_model = ResNet50(
    weights='imagenet',           # Vortrainierte Gewichte
    include_top=False,            # Obere Klassifikationsschichten entfernen
    input_shape=(32, 32, 3)       # CIFAR-10 Bildgröße
)

# Basis-Modell einfrieren (nicht trainieren)
base_model.trainable = False

print(f"\nBasis-Modell geladen: {base_model.name}")
print(f"Anzahl der Schichten: {len(base_model.layers)}")
print(f"Trainierbare Parameter: {base_model.trainable}")

In [ ]:
# Vollständiges Modell aufbauen mit Data Augmentation
print("\nBaue vollständiges Modell mit GPU-Optimierungen...")

# Input Layer explizit definieren für Mixed Precision
inputs = layers.Input(shape=(32, 32, 3), dtype='float32')

# Data Augmentation (nur während Training aktiv!)
x = data_augmentation(inputs)

# Basis-Modell (Feature Extractor)
x = base_model(x, training=False)

# Globales Average Pooling (Feature-Maps → Vektor)
x = layers.GlobalAveragePooling2D()(x)

# Dropout zur Regularisierung
x = layers.Dropout(0.3)(x)

# Versteckte Schicht 1
x = layers.Dense(128, activation='relu', name='hidden_1')(x)
x = layers.Dropout(0.2)(x)

# Versteckte Schicht 2
x = layers.Dense(64, activation='relu', name='hidden_2')(x)

# Output-Schicht (10 Klassen)
# dtype='float32' wichtig für Mixed Precision!
outputs = layers.Dense(10, activation='softmax', dtype='float32', name='output')(x)

# Modell erstellen
model = models.Model(inputs=inputs, outputs=outputs, name='CIFAR10_ResNet50_Optimized')

print("\n✅ Modell erfolgreich erstellt mit:")
print("   🔄 Data Augmentation (on-GPU)")
print("   🧠 ResNet50 Feature Extractor (frozen)")
print("   📊 Custom Classification Head")
print("   ⚡ Mixed Precision (float16/float32)")

In [ ]:
# Modell-Zusammenfassung
model.summary()

## 6. Modell kompilieren

In [ ]:
# Modell kompilieren
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',  # Für Integer-Labels
    metrics=['accuracy']
)

print("Modell kompiliert!")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Loss: sparse_categorical_crossentropy")
print(f"Metriken: accuracy")

## 7. Modell trainieren

In [ ]:
# Callbacks definieren
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

print("Callbacks konfiguriert:")
print("  - Early Stopping (patience=5)")
print("  - Learning Rate Reduction (patience=3)")

In [ ]:
# Training starten
print("\nStarte Training...\n")

# 🚀 M1 MAX GPU OPTIMIERUNG: Viel größere Batch Size dank Metal GPU!
# CPU: batch_size=32-64
# M1 Max Metal GPU: batch_size=128-256 (bessere GPU-Auslastung!)
BATCH_SIZE = 128  # Optimal für Metal GPU Performance

print(f"⚡ Batch Size: {BATCH_SIZE} (optimiert für M1 Max Metal GPU)")
print("="*60)

import time
start_time = time.time()

history = model.fit(
    train_images_normalized,
    train_labels_flat,
    epochs=20,
    batch_size=BATCH_SIZE,  # Große Batches für beste GPU-Auslastung
    validation_split=0.2,   # 20% für Validierung
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

training_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"⏱️  Training abgeschlossen in {training_time:.2f} Sekunden")
print(f"    ({training_time/60:.2f} Minuten)")
print(f"    Durchschnitt pro Epoche: {training_time/len(history.history['loss']):.2f}s")
print(f"{'='*60}")

In [ ]:
# 📊 GPU PERFORMANCE REPORT

print("\n" + "="*60)
print("📊 TRAINING PERFORMANCE REPORT")
print("="*60)

# Berechne Performance-Metriken
total_epochs = len(history.history['loss'])
total_samples = len(train_images_normalized)
samples_per_epoch = int(total_samples * 0.8)  # 80% Training (20% Validation)
total_samples_processed = samples_per_epoch * total_epochs

print(f"⏱️  Gesamtzeit: {training_time:.2f}s ({training_time/60:.2f} min)")
print(f"📈 Epochen: {total_epochs}")
print(f"🖼️  Samples pro Epoche: {samples_per_epoch:,}")
print(f"📊 Total Samples verarbeitet: {total_samples_processed:,}")
print(f"⚡ Durchsatz: {total_samples_processed/training_time:.1f} Bilder/Sekunde")
print(f"🔄 Zeit pro Epoche: {training_time/total_epochs:.2f}s")
print(f"📦 Batch Size: {BATCH_SIZE}")

# GPU Auslastung schätzen
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"\n🎮 GPU: {gpu_devices[0].name}")
    print(f"   ✅ Metal GPU wurde genutzt")
    print(f"   ⚡ Mixed Precision: Aktiv (float16)")
    print(f"   💾 Memory Growth: Aktiviert")
    
    # Performance-Vergleich
    estimated_cpu_time = training_time * 3.5  # GPU ist ~3-5x schneller
    print(f"\n🚀 Performance-Vergleich:")
    print(f"   GPU (Metal): {training_time/60:.1f} min")
    print(f"   CPU (geschätzt): {estimated_cpu_time/60:.1f} min")
    print(f"   Speedup: ~{estimated_cpu_time/training_time:.1f}x schneller")
else:
    print("\n⚠️  Warnung: Kein GPU-Training!")

print("="*60)

## 8. Training-Historie visualisieren

In [ ]:
# Training-Verlauf plotten
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Training', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    ax1.set_title('Modell Accuracy', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Loss
    ax2.plot(history.history['loss'], label='Training', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Validation', linewidth=2)
    ax2.set_title('Modell Loss', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history)

## 9. Modell auf Testdaten evaluieren

In [ ]:
# Evaluation auf Testdaten
print("Evaluiere Modell auf Testdaten...\n")

test_loss, test_accuracy = model.evaluate(
    test_images_normalized,
    test_labels_flat,
    verbose=1
)

print(f"\n{'='*50}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"{'='*50}")

## 10. Vorhersagen und Analyse

In [ ]:
# Vorhersagen auf Testdaten
print("Erstelle Vorhersagen...")
predictions = model.predict(test_images_normalized)
predicted_classes = np.argmax(predictions, axis=1)

print(f"Predictions Shape: {predictions.shape}")
print(f"Predicted Classes Shape: {predicted_classes.shape}")

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(test_labels_flat, predicted_classes)

# Visualisierung
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Anzahl'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Vorhergesagte Klasse', fontsize=12)
plt.ylabel('Wahre Klasse', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Detaillierter Classification Report
print("\nClassification Report:")
print("="*70)
report = classification_report(test_labels_flat, predicted_classes, 
                               target_names=class_names, digits=4)
print(report)

In [ ]:
# Pro-Klasse Accuracy visualisieren
class_accuracy = []
for i in range(10):
    mask = test_labels_flat == i
    class_acc = (predicted_classes[mask] == test_labels_flat[mask]).mean()
    class_accuracy.append(class_acc)

plt.figure(figsize=(12, 6))
bars = plt.bar(range(10), class_accuracy)
plt.xticks(range(10), class_names, rotation=45, ha='right')
plt.xlabel('Klasse')
plt.ylabel('Accuracy')
plt.title('Accuracy pro Klasse', fontsize=14, fontweight='bold')
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)

# Werte auf Balken anzeigen
for i, (bar, acc) in enumerate(zip(bars, class_accuracy)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2%}', ha='center', va='bottom', fontsize=9)
    bar.set_alpha(0.8)

plt.tight_layout()
plt.show()

# Beste und schlechteste Klassen
best_idx = np.argmax(class_accuracy)
worst_idx = np.argmin(class_accuracy)
print(f"\nBeste Klasse: {class_names[best_idx]} ({class_accuracy[best_idx]:.2%})")
print(f"Schlechteste Klasse: {class_names[worst_idx]} ({class_accuracy[worst_idx]:.2%})")

## 11. Beispielvorhersagen visualisieren

In [ ]:
# Korrekte Vorhersagen zeigen
def plot_predictions(images, true_labels, pred_labels, class_names, n=12, correct=True):
    """
    Zeigt Beispiele von korrekten oder falschen Vorhersagen
    """
    if correct:
        mask = (pred_labels == true_labels)
        title = "Korrekte Vorhersagen"
        color = 'green'
    else:
        mask = (pred_labels != true_labels)
        title = "Falsche Vorhersagen"
        color = 'red'
    
    indices = np.where(mask)[0]
    if len(indices) == 0:
        print(f"Keine {'korrekten' if correct else 'falschen'} Vorhersagen gefunden!")
        return
    
    sample_indices = np.random.choice(indices, min(n, len(indices)), replace=False)
    
    plt.figure(figsize=(15, 12))
    for i, idx in enumerate(sample_indices):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[idx])
        true_class = class_names[true_labels[idx]]
        pred_class = class_names[pred_labels[idx]]
        
        if correct:
            plt.title(f'Wahr: {true_class}\nPred: {pred_class}', 
                     color=color, fontsize=9)
        else:
            plt.title(f'Wahr: {true_class}\nPred: {pred_class}', 
                     color=color, fontsize=9, fontweight='bold')
        plt.axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

# Korrekte Vorhersagen
plot_predictions(test_images, test_labels_flat, predicted_classes, 
                class_names, n=12, correct=True)

In [ ]:
# Falsche Vorhersagen
plot_predictions(test_images, test_labels_flat, predicted_classes, 
                class_names, n=12, correct=False)

## 12. Modell speichern

In [ ]:
# Modell speichern
model_path = 'cifar10_resnet50_model.keras'
model.save(model_path)
print(f"Modell gespeichert unter: {model_path}")

# Alternative: Nur Gewichte speichern
weights_path = 'cifar10_resnet50.weights.h5'
model.save_weights(weights_path)
print(f"Gewichte gespeichert unter: {weights_path}")

## 13. Verbesserungsvorschläge

### Für bessere Performance:

1. **Mehr Daten**: Nutze alle 50.000 Trainingsbilder statt 10.000

2. **Data Augmentation**: Erweitere Daten durch Transformationen
   ```python
   from tensorflow.keras.preprocessing.image import ImageDataGenerator
   
   datagen = ImageDataGenerator(
       rotation_range=15,
       width_shift_range=0.1,
       height_shift_range=0.1,
       horizontal_flip=True
   )
   ```

3. **Fine-Tuning**: Trainiere auch einige Schichten des Basis-Modells
   ```python
   base_model.trainable = True
   # Friere nur die ersten N Schichten ein
   for layer in base_model.layers[:-20]:
       layer.trainable = False
   ```

4. **Andere Architekturen testen**:
   - EfficientNet
   - MobileNet
   - VGG16

5. **Hyperparameter-Tuning**:
   - Learning Rate anpassen
   - Batch Size variieren
   - Mehr/weniger versteckte Schichten
   - Dropout-Rate anpassen

6. **Ensemble Methods**: Kombiniere mehrere Modelle

## 🚀 VERBESSERTES MODELL - Optimiert für CIFAR-10

**Problem mit ResNet50:** Zu tief für 32x32 Bilder → niedrige Accuracy

**Lösung:** Custom CNN speziell für kleine Bilder designed → **70-85% Accuracy**

In [ ]:
# 🎯 CUSTOM CNN - Speziell für 32x32 CIFAR-10 Bilder
print("="*60)
print("Baue Custom CNN optimiert für CIFAR-10...")
print("="*60)

# Input Layer
inputs_v2 = layers.Input(shape=(32, 32, 3), dtype='float32')

# Data Augmentation (on-GPU!)
x = data_augmentation(inputs_v2)

# Block 1: 32x32 → 16x16
x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.2)(x)

# Block 2: 16x16 → 8x8
x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.3)(x)

# Block 3: 8x8 → 4x4
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.4)(x)

# Global Average Pooling
x = layers.GlobalAveragePooling2D()(x)

# Dense Layers
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

# Output Layer (WICHTIG: dtype='float32' für Mixed Precision!)
outputs_v2 = layers.Dense(10, activation='softmax', dtype='float32')(x)

# Modell erstellen
model_v2 = models.Model(inputs=inputs_v2, outputs=outputs_v2, name='CIFAR10_CustomCNN')

print("\n✅ Custom CNN erstellt!")
print(f"   📊 Architektur: 3 Conv Blocks + 2 Dense Layers")
print(f"   🔄 Data Augmentation: Aktiviert")
print(f"   ⚡ BatchNormalization: Alle Layer")
print(f"   📉 Dropout: 0.2 → 0.5 (progressiv)")

# Modell-Summary
model_v2.summary()

# Anzahl Parameter vergleichen
print(f"\n" + "="*60)
print("📊 MODELL-VERGLEICH")
print("="*60)
print(f"ResNet50 Transfer Learning:")
print(f"  - Total Params: 23.86M")
print(f"  - Trainable: 271K (nur 1.1%)")
print(f"  - Test Accuracy: ~26%")
print(f"\nCustom CNN:")
print(f"  - Total Params: {model_v2.count_params()/1e6:.2f}M")
print(f"  - Trainable: {model_v2.count_params()/1e6:.2f}M (100%)")
print(f"  - Erwartete Accuracy: 70-85%")
print("="*60)

In [ ]:
# Modell kompilieren
model_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Custom CNN kompiliert!")
print(f"   Optimizer: Adam (lr=0.001)")
print(f"   Loss: sparse_categorical_crossentropy")
print(f"   Metrics: accuracy")

In [ ]:
# Training des Custom CNN
print("\n🚀 Starte Training des Custom CNN...\n")
print(f"⚡ Batch Size: {BATCH_SIZE}")
print("="*60)

import time
start_time_v2 = time.time()

history_v2 = model_v2.fit(
    train_images_normalized,
    train_labels_flat,
    epochs=30,  # Mehr Epochen für bessere Konvergenz
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

training_time_v2 = time.time() - start_time_v2

print(f"\n{'='*60}")
print(f"⏱️  Training abgeschlossen in {training_time_v2:.2f}s ({training_time_v2/60:.2f} min)")
print(f"    Durchschnitt pro Epoche: {training_time_v2/len(history_v2.history['loss']):.2f}s")
print(f"    Durchsatz: {(len(train_images_normalized)*0.8*len(history_v2.history['loss'])/training_time_v2):.1f} Bilder/Sek")
print(f"{'='*60}")

In [ ]:
# Custom CNN evaluieren
print("\n📊 Evaluiere Custom CNN auf Testdaten...\n")

test_loss_v2, test_accuracy_v2 = model_v2.evaluate(
    test_images_normalized,
    test_labels_flat,
    verbose=1
)

print(f"\n{'='*60}")
print("🎯 ERGEBNISSE - CUSTOM CNN")
print(f"{'='*60}")
print(f"Test Loss: {test_loss_v2:.4f}")
print(f"Test Accuracy: {test_accuracy_v2:.4f} ({test_accuracy_v2*100:.2f}%)")
print(f"{'='*60}")

# Vergleich mit ResNet50
print(f"\n{'='*60}")
print("📊 MODELL-VERGLEICH: ERGEBNISSE")
print(f"{'='*60}")
print(f"\n1️⃣  ResNet50 Transfer Learning:")
print(f"   Test Accuracy: 26.20%")
print(f"   Training Zeit: {training_time/60:.1f} min")
print(f"   Parameter: 23.86M (nur 1.1% trainierbar)")

print(f"\n2️⃣  Custom CNN:")
print(f"   Test Accuracy: {test_accuracy_v2*100:.2f}%")
print(f"   Training Zeit: {training_time_v2/60:.1f} min")
print(f"   Parameter: {model_v2.count_params()/1e6:.2f}M (100% trainierbar)")

improvement = ((test_accuracy_v2 - 0.2620) / 0.2620) * 100
print(f"\n✅ Verbesserung: +{improvement:.1f}%")
print(f"   ({test_accuracy_v2*100:.2f}% vs 26.20%)")
print(f"{'='*60}")

In [ ]:
# 1️⃣ Training History - Accuracy & Loss Kurven
print("📈 Training History Visualisierung\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Accuracy über Zeit
axes[0, 0].plot(history_v2.history['accuracy'], label='Training', linewidth=2, marker='o', markersize=4)
axes[0, 0].plot(history_v2.history['val_accuracy'], label='Validation', linewidth=2, marker='s', markersize=4)
axes[0, 0].set_title('Custom CNN - Accuracy über Epochen', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].set_ylim([0, 1])

# Loss über Zeit
axes[0, 1].plot(history_v2.history['loss'], label='Training', linewidth=2, marker='o', markersize=4)
axes[0, 1].plot(history_v2.history['val_loss'], label='Validation', linewidth=2, marker='s', markersize=4)
axes[0, 1].set_title('Custom CNN - Loss über Epochen', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Learning Rate über Zeit
axes[1, 0].plot(history_v2.history['learning_rate'], linewidth=2, marker='o', markersize=4, color='green')
axes[1, 0].set_title('Learning Rate Reduktion', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(alpha=0.3)

# Vergleich: Training vs Validation Accuracy
diff = [abs(t - v) for t, v in zip(history_v2.history['accuracy'], history_v2.history['val_accuracy'])]
axes[1, 1].plot(diff, linewidth=2, marker='o', markersize=4, color='red')
axes[1, 1].set_title('Overfitting Indikator (Train-Val Gap)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('|Train Acc - Val Acc|')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].axhline(y=0.05, color='orange', linestyle='--', label='Akzeptabler Gap (5%)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"\n✅ Beste Validation Accuracy: {max(history_v2.history['val_accuracy'])*100:.2f}%")
print(f"✅ Finale Test Accuracy: {test_accuracy_v2*100:.2f}%")
print(f"✅ Overfitting Gap: {abs(history_v2.history['accuracy'][-1] - history_v2.history['val_accuracy'][-1])*100:.2f}%")

### 📊 Custom CNN - Detaillierte Visualisierungen

In [ ]:
# 📊 Per-Class Accuracy Analysis
from sklearn.metrics import f1_score

# Berechne Per-Class Metriken
class_accuracies = []
class_f1_scores = []

for i in range(10):
    # Accuracy für diese Klasse
    mask = test_labels_flat == i
    if mask.sum() > 0:
        acc = (predicted_classes_v2[mask] == i).sum() / mask.sum() * 100
        class_accuracies.append(acc)
        
        # F1-Score für diese Klasse
        y_true_binary = (test_labels_flat == i).astype(int)
        y_pred_binary = (predicted_classes_v2 == i).astype(int)
        f1 = f1_score(y_true_binary, y_pred_binary) * 100
        class_f1_scores.append(f1)
    else:
        class_accuracies.append(0)
        class_f1_scores.append(0)

# Visualisierung
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('📊 Per-Class Performance - Custom CNN', fontsize=18, fontweight='bold')

# Accuracy pro Klasse
ax1 = axes[0]
colors = ['green' if acc > 75 else 'orange' if acc > 60 else 'red' for acc in class_accuracies]
bars1 = ax1.bar(class_names, class_accuracies, color=colors, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Klasse', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax1.set_title('Accuracy pro Klasse', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=np.mean(class_accuracies), color='blue', linestyle='--', linewidth=2, label=f'Durchschnitt: {np.mean(class_accuracies):.1f}%')
ax1.legend()

# Werte auf Balken anzeigen
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# F1-Score pro Klasse
ax2 = axes[1]
colors_f1 = ['green' if f1 > 75 else 'orange' if f1 > 60 else 'red' for f1 in class_f1_scores]
bars2 = ax2.bar(class_names, class_f1_scores, color=colors_f1, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Klasse', fontsize=12, fontweight='bold')
ax2.set_ylabel('F1-Score (%)', fontsize=12, fontweight='bold')
ax2.set_title('F1-Score pro Klasse', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=np.mean(class_f1_scores), color='blue', linestyle='--', linewidth=2, label=f'Durchschnitt: {np.mean(class_f1_scores):.1f}%')
ax2.legend()

# Werte auf Balken anzeigen
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Statistiken ausgeben
print("\n" + "="*80)
print("📊 PER-CLASS STATISTIKEN")
print("="*80)
for i, name in enumerate(class_names):
    print(f"{name:12s}: Accuracy = {class_accuracies[i]:5.1f}%  |  F1-Score = {class_f1_scores[i]:5.1f}%")
print("="*80)
print(f"Beste Klasse:      {class_names[np.argmax(class_accuracies)]} ({max(class_accuracies):.1f}%)")
print(f"Schlechteste Klasse: {class_names[np.argmin(class_accuracies)]} ({min(class_accuracies):.1f}%)")
print(f"Durchschnitt:      {np.mean(class_accuracies):.1f}% ± {np.std(class_accuracies):.1f}%")
print("="*80)

In [ ]:
# 🎨 Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report

cm_v2 = confusion_matrix(test_labels_flat, predicted_classes_v2)

# Visualisierung mit zwei Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('📊 Confusion Matrix - Custom CNN', fontsize=18, fontweight='bold')

# Absolute Zahlen
ax1 = axes[0]
im1 = ax1.imshow(cm_v2, cmap='Blues', aspect='auto')
ax1.set_xlabel('Vorhergesagte Klasse', fontsize=12, fontweight='bold')
ax1.set_ylabel('Wahre Klasse', fontsize=12, fontweight='bold')
ax1.set_title('Absolute Zahlen', fontsize=14, fontweight='bold')
ax1.set_xticks(range(10))
ax1.set_yticks(range(10))
ax1.set_xticklabels(class_names, rotation=45, ha='right')
ax1.set_yticklabels(class_names)

# Zahlen in Zellen eintragen
for i in range(10):
    for j in range(10):
        text = ax1.text(j, i, cm_v2[i, j], ha='center', va='center',
                       color='white' if cm_v2[i, j] > cm_v2.max()/2 else 'black',
                       fontsize=10, fontweight='bold')

plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

# Normalisiert (Prozente pro Klasse)
cm_normalized = cm_v2.astype('float') / cm_v2.sum(axis=1)[:, np.newaxis] * 100
ax2 = axes[1]
im2 = ax2.imshow(cm_normalized, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
ax2.set_xlabel('Vorhergesagte Klasse', fontsize=12, fontweight='bold')
ax2.set_ylabel('Wahre Klasse', fontsize=12, fontweight='bold')
ax2.set_title('Normalisiert (% pro Klasse)', fontsize=14, fontweight='bold')
ax2.set_xticks(range(10))
ax2.set_yticks(range(10))
ax2.set_xticklabels(class_names, rotation=45, ha='right')
ax2.set_yticklabels(class_names)

# Prozente in Zellen eintragen
for i in range(10):
    for j in range(10):
        text = ax2.text(j, i, f'{cm_normalized[i, j]:.1f}%', ha='center', va='center',
                       color='white' if cm_normalized[i, j] < 50 else 'black',
                       fontsize=9, fontweight='bold')

plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label='Genauigkeit (%)')

plt.tight_layout()
plt.show()

# Classification Report
print("\n" + "="*80)
print("📋 CLASSIFICATION REPORT - Custom CNN")
print("="*80)
print(classification_report(test_labels_flat, predicted_classes_v2, target_names=class_names))
print("="*80)

In [ ]:
# 🎯 Vorhersagen generieren
print("Generiere Vorhersagen für Test-Set...")
predictions_v2 = model_v2.predict(test_images_normalized, verbose=0)
predicted_classes_v2 = np.argmax(predictions_v2, axis=1)
print(f"✅ Vorhersagen für {len(test_images_normalized)} Testbilder generiert")

In [ ]:
# 📈 Training History Visualisierung
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('📊 Custom CNN Training History', fontsize=18, fontweight='bold')

# 1. Accuracy
ax1 = axes[0, 0]
ax1.plot(history_v2.history['accuracy'], label='Training Accuracy', linewidth=2, marker='o', markersize=4)
ax1.plot(history_v2.history['val_accuracy'], label='Validation Accuracy', linewidth=2, marker='s', markersize=4)
ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 2. Loss
ax2 = axes[0, 1]
ax2.plot(history_v2.history['loss'], label='Training Loss', linewidth=2, marker='o', markersize=4, color='orange')
ax2.plot(history_v2.history['val_loss'], label='Validation Loss', linewidth=2, marker='s', markersize=4, color='red')
ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax2.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# 3. Learning Rate
ax3 = axes[1, 0]
if 'lr' in history_v2.history:
    ax3.plot(history_v2.history['lr'], linewidth=2, marker='d', markersize=4, color='green')
    ax3.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
    ax3.set_title('Learning Rate Reduction', fontsize=14, fontweight='bold')
    ax3.set_yscale('log')
    ax3.grid(True, alpha=0.3, which='both')
else:
    ax3.text(0.5, 0.5, 'Learning Rate nicht verfügbar', ha='center', va='center', fontsize=12)
    ax3.axis('off')

# 4. Overfitting Gap
ax4 = axes[1, 1]
train_acc = history_v2.history['accuracy']
val_acc = history_v2.history['val_accuracy']
overfitting_gap = [t - v for t, v in zip(train_acc, val_acc)]
epochs = range(1, len(train_acc) + 1)

ax4.plot(epochs, overfitting_gap, linewidth=2, marker='o', markersize=4, color='purple')
ax4.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax4.fill_between(epochs, 0, overfitting_gap, alpha=0.3, color='purple')
ax4.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax4.set_ylabel('Gap (Train - Val Accuracy)', fontsize=12, fontweight='bold')
ax4.set_title('Overfitting Indicator', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Trainingszusammenfassung:")
print(f"   Beste Val Accuracy: {max(history_v2.history['val_accuracy'])*100:.2f}% (Epoch {history_v2.history['val_accuracy'].index(max(history_v2.history['val_accuracy']))+1})")
print(f"   Finale Val Accuracy: {history_v2.history['val_accuracy'][-1]*100:.2f}%")
print(f"   Finale Val Loss: {history_v2.history['val_loss'][-1]:.4f}")

---

## 🏆 MODELL-VERGLEICH: Alle Top-Architekturen

Jetzt testen wir **4 verschiedene Modelle** und vergleichen die Ergebnisse:

1. **EfficientNet-B0** - State-of-the-art Effizienz
2. **MobileNetV2** - Maximale Geschwindigkeit
3. **Custom ResNet** - ResNet für kleine Bilder
4. **DenseNet121** - Dense Connections

**Ziel:** Finde das beste Modell für CIFAR-10! 🎯

In [ ]:
# 🎯 Hilfsfunktionen für Modell-Vergleich
import time

# Dictionary für alle Ergebnisse
model_results = {}

def train_and_evaluate_model(model, model_name, epochs=25):
    """
    Trainiert und evaluiert ein Modell
    Returns: (test_accuracy, test_loss, training_time, history)
    """
    print(f"\n{'='*60}")
    print(f"🚀 Training: {model_name}")
    print(f"{'='*60}\n")
    
    # Kompilieren
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Training
    start_time = time.time()
    history = model.fit(
        train_images_normalized,
        train_labels_flat,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    training_time = time.time() - start_time
    
    # Evaluation
    test_loss, test_accuracy = model.evaluate(
        test_images_normalized,
        test_labels_flat,
        verbose=0
    )
    
    # Ergebnisse speichern
    model_results[model_name] = {
        'test_accuracy': test_accuracy,
        'test_loss': test_loss,
        'training_time': training_time,
        'epochs_trained': len(history.history['loss']),
        'best_val_accuracy': max(history.history['val_accuracy']),
        'params': model.count_params(),
        'history': history
    }
    
    print(f"\n{'='*60}")
    print(f"✅ {model_name} - Fertig!")
    print(f"   Test Accuracy: {test_accuracy*100:.2f}%")
    print(f"   Training Zeit: {training_time/60:.1f} min")
    print(f"   Epochen: {len(history.history['loss'])}")
    print(f"{'='*60}\n")
    
    return test_accuracy, test_loss, training_time, history

print("✅ Hilfsfunktionen geladen!")
print("   Bereit für Modell-Vergleich")

In [ ]:
# EfficientNet-B0 Model
from tensorflow.keras.applications import EfficientNetB0

print("Erstelle EfficientNet-B0 Modell...")

# Base Model
base_efficientnet = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(32, 32, 3)
)
base_efficientnet.trainable = False

# Input
inputs_eff = layers.Input(shape=(32, 32, 3), dtype='float32')
x = data_augmentation(inputs_eff)

# EfficientNet Base
x = base_efficientnet(x, training=False)

# Classification Head
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs_eff = layers.Dense(10, activation='softmax', dtype='float32')(x)

model_efficientnet = models.Model(inputs=inputs_eff, outputs=outputs_eff, name='EfficientNet_B0')

print(f"✅ EfficientNet-B0 erstellt!")
print(f"   Parameter: {model_efficientnet.count_params()/1e6:.2f}M")

# Training
train_and_evaluate_model(model_efficientnet, "EfficientNet-B0", epochs=25)

### 1️⃣ EfficientNet-B0 - State-of-the-Art Effizienz

In [ ]:
# DenseNet121 Training
model_results['DenseNet121'] = train_and_evaluate_model(
    model_densenet,
    'DenseNet121',
    train_images_normalized,
    train_labels_flat,
    test_images_normalized,
    test_labels_flat,
    batch_size=BATCH_SIZE,
    epochs=30
)

In [ ]:
# 📈 Training History Vergleich - Alle Modelle
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('📈 Training History: Alle Modelle im Vergleich', fontsize=18, fontweight='bold')

colors_models = ['blue', 'green', 'red', 'orange', 'purple']

# Validation Accuracy
ax1 = axes[0]
for i, (model_name, row) in enumerate(comparison_df.iterrows()):
    history = row['history']
    epochs_count = range(1, len(history.history['val_accuracy']) + 1)
    ax1.plot(epochs_count, history.history['val_accuracy'], 
            label=model_name, linewidth=2, marker='o', markersize=3, 
            color=colors_models[i], alpha=0.8)

ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax1.set_ylabel('Validation Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Validation Accuracy über Epochen', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10, loc='lower right')
ax1.grid(True, alpha=0.3)

# Validation Loss
ax2 = axes[1]
for i, (model_name, row) in enumerate(comparison_df.iterrows()):
    history = row['history']
    epochs_count = range(1, len(history.history['val_loss']) + 1)
    ax2.plot(epochs_count, history.history['val_loss'], 
            label=model_name, linewidth=2, marker='o', markersize=3, 
            color=colors_models[i], alpha=0.8)

ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax2.set_ylabel('Validation Loss', fontsize=12, fontweight='bold')
ax2.set_title('Validation Loss über Epochen', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 📊 Visualisierung: Modell-Vergleich
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('🏆 Modell-Vergleich: Alle 5 Architekturen', fontsize=18, fontweight='bold')

# 1. Test Accuracy Vergleich
ax1 = axes[0, 0]
models = comparison_df.index
accuracies = comparison_df['test_accuracy'] * 100
colors = ['green' if acc >= 75 else 'orange' if acc >= 60 else 'red' for acc in accuracies]

bars = ax1.barh(models, accuracies, color=colors, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax1.set_title('Test Accuracy Vergleich', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Werte auf Balken
for bar in bars:
    width = bar.get_width()
    ax1.text(width + 1, bar.get_y() + bar.get_height()/2, 
            f'{width:.1f}%', va='center', fontweight='bold')

# 2. Training Zeit Vergleich
ax2 = axes[0, 1]
training_times = comparison_df['training_time'] / 60  # In Minuten
bars2 = ax2.barh(models, training_times, color='steelblue', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Trainingszeit (Minuten)', fontsize=12, fontweight='bold')
ax2.set_title('Trainingszeit Vergleich', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

for bar in bars2:
    width = bar.get_width()
    ax2.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{width:.1f} min', va='center', fontweight='bold')

# 3. Accuracy vs Training Zeit (Scatter)
ax3 = axes[1, 0]
ax3.scatter(training_times, accuracies, s=300, c=colors, edgecolors='black', alpha=0.7, linewidths=2)

# Annotationen
for i, model in enumerate(models):
    ax3.annotate(model, (training_times[i], accuracies[i]), 
                fontsize=9, fontweight='bold', ha='center', va='bottom')

ax3.set_xlabel('Trainingszeit (Minuten)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax3.set_title('Effizienz: Accuracy vs. Trainingszeit', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Loss Vergleich
ax4 = axes[1, 1]
losses = comparison_df['test_loss']
bars4 = ax4.barh(models, losses, color='coral', edgecolor='black', alpha=0.7)
ax4.set_xlabel('Test Loss', fontsize=12, fontweight='bold')
ax4.set_title('Test Loss Vergleich (niedriger = besser)', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='x')

for bar in bars4:
    width = bar.get_width()
    ax4.text(width + 0.05, bar.get_y() + bar.get_height()/2, 
            f'{width:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 📊 Vergleichstabelle erstellen
print("="*100)
print("🏆 MODELL-VERGLEICH: ALLE 5 ARCHITEKTUREN")
print("="*100)

# DataFrame erstellen
comparison_df = pd.DataFrame(model_results).T
comparison_df = comparison_df.sort_values('test_accuracy', ascending=False)

# Formatierte Ausgabe
print(f"\n{'Modell':<25} {'Accuracy':>10} {'Loss':>10} {'Zeit (min)':>12} {'Epochen':>10}")
print("-"*100)

for model_name, row in comparison_df.iterrows():
    acc = row['test_accuracy'] * 100
    loss = row['test_loss']
    time_min = row['training_time'] / 60
    epochs = len(row['history'].history['loss'])
    
    # Emoji basierend auf Accuracy
    if acc >= 75:
        emoji = "🥇"
    elif acc >= 60:
        emoji = "🥈"
    elif acc >= 50:
        emoji = "🥉"
    else:
        emoji = "❌"
    
    print(f"{emoji} {model_name:<23} {acc:>9.2f}% {loss:>10.4f} {time_min:>11.1f} {epochs:>10}")

print("="*100)

# Beste Modell hervorheben
best_model = comparison_df.index[0]
best_acc = comparison_df.iloc[0]['test_accuracy'] * 100
print(f"\n🏆 BESTES MODELL: {best_model} mit {best_acc:.2f}% Test Accuracy")
print("="*100)

---
## 📊 Finaler Modell-Vergleich

Vergleichen wir alle 5 Modelle:

In [ ]:
# DenseNet121 Model
from tensorflow.keras.applications import DenseNet121

print("Erstelle DenseNet121 Modell...")

# Basis-Modell
base_densenet = DenseNet121(
    input_shape=(32, 32, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

# Friere Basis-Layers ein
base_densenet.trainable = False

# Vollständiges Modell
inputs_dense = layers.Input(shape=(32, 32, 3), dtype='float32')
x = data_augmentation(inputs_dense)
x = base_densenet(x, training=False)
x = layers.Dropout(0.4)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs_dense = layers.Dense(10, activation='softmax', dtype='float32')(x)

model_densenet = models.Model(inputs=inputs_dense, outputs=outputs_dense, name='CIFAR10_DenseNet121')

# Kompilieren
model_densenet.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\n✅ DenseNet121 Modell erstellt!")
print(f"   Trainierbare Parameter: {model_densenet.count_params():,}")
model_densenet.summary()

### 4️⃣ DenseNet121 - Dense Connections

In [ ]:
# Custom ResNet Training
model_results['CustomResNet'] = train_and_evaluate_model(
    model_custom_resnet,
    'CustomResNet',
    train_images_normalized,
    train_labels_flat,
    test_images_normalized,
    test_labels_flat,
    batch_size=BATCH_SIZE,
    epochs=30
)

In [ ]:
# Custom ResNet with Residual Blocks
print("Erstelle Custom ResNet mit Residual Blocks...")

def residual_block(x, filters, kernel_size=3, stride=1, activation='relu'):
    """Ein ResNet Residual Block"""
    # Shortcut
    shortcut = x
    
    # Main Path
    x = layers.Conv2D(filters, kernel_size, strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    
    x = layers.Conv2D(filters, kernel_size, strides=1, padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Wenn Shape sich ändert, passe Shortcut an
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add Shortcut (Residual Connection)
    x = layers.Add()([x, shortcut])
    x = layers.Activation(activation)(x)
    
    return x

# Input Layer
inputs_resnet = layers.Input(shape=(32, 32, 3), dtype='float32')
x = data_augmentation(inputs_resnet)

# Initial Conv Layer
x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)

# Residual Blocks - Stage 1 (32x32)
x = residual_block(x, 64)
x = residual_block(x, 64)

# Residual Blocks - Stage 2 (16x16)
x = residual_block(x, 128, stride=2)
x = residual_block(x, 128)

# Residual Blocks - Stage 3 (8x8)
x = residual_block(x, 256, stride=2)
x = residual_block(x, 256)

# Residual Blocks - Stage 4 (4x4)
x = residual_block(x, 512, stride=2)
x = residual_block(x, 512)

# Classification Head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs_resnet = layers.Dense(10, activation='softmax', dtype='float32')(x)

model_custom_resnet = models.Model(inputs=inputs_resnet, outputs=outputs_resnet, name='CIFAR10_CustomResNet')

# Kompilieren
model_custom_resnet.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\n✅ Custom ResNet Modell erstellt!")
print(f"   Trainierbare Parameter: {model_custom_resnet.count_params():,}")
model_custom_resnet.summary()

### 3️⃣ Custom ResNet - ResNet für kleine Bilder

In [ ]:
# MobileNetV2 Training
model_results['MobileNetV2'] = train_and_evaluate_model(
    model_mobilenet,
    'MobileNetV2',
    train_images_normalized,
    train_labels_flat,
    test_images_normalized,
    test_labels_flat,
    batch_size=BATCH_SIZE,
    epochs=30
)

In [ ]:
# MobileNetV2 Model
from tensorflow.keras.applications import MobileNetV2

print("Erstelle MobileNetV2 Modell...")

# Basis-Modell
base_mobilenet = MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

# Friere Basis-Layers ein
base_mobilenet.trainable = False

# Vollständiges Modell
inputs_mobile = layers.Input(shape=(32, 32, 3), dtype='float32')
x = data_augmentation(inputs_mobile)
x = base_mobilenet(x, training=False)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs_mobile = layers.Dense(10, activation='softmax', dtype='float32')(x)

model_mobilenet = models.Model(inputs=inputs_mobile, outputs=outputs_mobile, name='CIFAR10_MobileNetV2')

# Kompilieren
model_mobilenet.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\n✅ MobileNetV2 Modell erstellt!")
print(f"   Trainierbare Parameter: {model_mobilenet.count_params():,}")
model_mobilenet.summary()

### 2️⃣ MobileNetV2 - Maximale Geschwindigkeit